# Beyond Views: What Makes Users Stop Scrolling?

A reproducible product analytics walkthrough using synthetic short-form video events. **No TikTok or real-user data is used.**


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
DATA = Path('../data')
df = pd.read_csv(DATA/'synthetic_user_events.csv', parse_dates=['event_date'])
df.head()


## Metric design
Raw watch seconds favor longer videos. We therefore use watch ratio, completion, quick swipes, high-intent actions, negative feedback, and session continuation together.


In [ ]:
metrics = df.groupby('content_category').agg(
 views=('event_id','size'), avg_watch_seconds=('watch_time_seconds','mean'),
 avg_watch_ratio=('watch_ratio','mean'), completion_rate=('completed_video','mean'),
 quick_swipe_rate=('quick_swipe','mean'), share_rate=('shared','mean'),
 follow_rate=('followed_creator','mean'), negative_feedback_rate=('negative_feedback','mean'),
 session_continuation_rate=('continued_session','mean')).sort_values('avg_watch_ratio', ascending=False)
metrics.style.format('{:.1%}', subset=[c for c in metrics if c.endswith('rate') or c=='avg_watch_ratio'])


In [ ]:
ax = metrics.sort_values('avg_watch_ratio').avg_watch_ratio.plot.barh(figsize=(9,5), color='#25F4EE')
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.set(title='Average share of video watched', xlabel='Watch ratio', ylabel='')
plt.tight_layout()


## Signal diagnosis
Correlation does not establish causation. This step identifies candidates for an experiment rather than claiming that any action causes continued use.


In [ ]:
signals = ['watch_ratio','completed_video','quick_swipe','liked','shared','followed_creator','rewatched','negative_feedback']
df[signals+['continued_session']].corr()['continued_session'].drop('continued_session').sort_values()


## Recommendation and experiment
Test a multi-signal ranking score against a watch-time-led control. Use meaningful-view rate and continuation as primary outcomes; protect negative feedback, content diversity, creator concentration, and extreme session length as guardrails.
